[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/helloash-17/Prompt-Basics/blob/main/promptAdvanced.ipynb)

# Prompt Engineering: Session 2
### Hands-on session

## Quick recap of Session 1
- A **vague** question gave a so-so answer. A **specific** question gave a much better one.
- Adding **"think step by step"** made the AI solve the maths problem correctly instead of guessing.

Today we reuse those same two ideas on real tasks you'll actually do, plus two new tricks: getting different **perspectives** on the same code, and **refining** an answer over a few turns instead of expecting it perfect in one shot.

Run every cell top to bottom, and try the exercise cells yourself.

In [ ]:
# Setup (same as Session 1)
!pip install groq --quiet

In [ ]:
from groq import Groq

try:
    from google.colab import userdata
    api_key = userdata.get("GROQ_API_KEY")
except Exception:
    import getpass
    api_key = getpass.getpass("Paste your Groq API key here: ")

client = Groq(api_key=api_key)
MODEL = "openai/gpt-oss-20b"

def ask(prompt, system=None):
    """Same ask() as Session 1, with one new optional part: system.
    system sets the AI's role/persona. We'll use it in Section 3."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    response = client.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

print("Connected! ask() is ready.")

## 1. Chain of Thought — for Debugging

In Session 1, "think step by step" fixed a maths problem. Same trick works on **debugging**: instead of just saying "fix this", ask the AI to first explain the code, then find the bug, then fix it. This stops it from guessing.

In [ ]:
buggy_code = """
def average(numbers):
    total = 0
    for n in numbers:
        total = n
    return total / len(numbers)
"""

prompt = "Think step by step. First explain what this code does. " \
         "Then identify where it might be going wrong. " \
         "Then suggest a fix.\n\nCode:\n" + buggy_code

print(ask(prompt))

**Exercise:** Paste one of your own buggy functions into `my_buggy_code` below and run the cell.

In [ ]:
my_buggy_code = """
# paste your buggy code here
"""

prompt = "Think step by step. First explain what this code does. " \
         "Then identify where it might be going wrong. " \
         "Then suggest a fix.\n\nCode:\n" + my_buggy_code

print(ask(prompt))

## 2. Few-Shot Prompting — for Controlling Format

In Session 1, showing 2 examples turned casual statements into formal ones, in exactly the style shown. This is the most reliable way to control output format — more reliable than just describing the format in words.

Below, we show 2 examples of adding a short docstring to a function, then let the AI do the 3rd one the same way.

In [ ]:
prompt = """Add a one-line docstring to each function, in the same style as the examples.

def add(a, b):
    return a + b

def add(a, b):
    \"\"\"Adds two numbers and returns the result.\"\"\"
    return a + b

def is_even(n):
    return n % 2 == 0

def is_even(n):
    \"\"\"Returns True if n is even.\"\"\"
    return n % 2 == 0

def max_of_three(a, b, c):
    return max(a, b, c)

def max_of_three(a, b, c):"""

print(ask(prompt))

**Exercise:** Change the last function (`max_of_three`) to one of your own. Keep the two examples above it — they teach the AI the style to copy.

In [ ]:
# Copy the prompt above, keep the two examples, change only the last function.


## 3. Role Prompting — Different Perspectives on the Same Code

The `system` message tells the AI who to "be" before it answers. Same question, same code — but a different persona changes what it focuses on:
- A **security engineer** looks for vulnerabilities.
- A **junior developer** explains things simply.
- A **tech lead** thinks about maintainability.

Run all three cells below and compare the answers — same code every time, only the persona changes.

In [ ]:
code_to_review = """
def get_user(user_id):
    query = "SELECT * FROM users WHERE id = " + user_id
    return db.execute(query)
"""

print(ask("Review this code:\n" + code_to_review,
           system="You are a security engineer. Focus only on vulnerabilities."))

In [ ]:
print(ask("Review this code:\n" + code_to_review,
           system="You are a junior developer. Explain the code simply."))

In [ ]:
print(ask("Review this code:\n" + code_to_review,
           system="You are a tech lead. Focus on maintainability."))

**Exercise:** Copy one of the three cells above and change the `system` text to a persona of your own choice (e.g. "You are a QA tester.").

## 4. Iterative Refinement — Don't Expect Perfection in One Shot

Every `ask()` so far has been a fresh question — the AI forgets it right after. Real prompting is usually a back-and-forth: get a first answer, then ask for corrections, like reviewing a teammate's first draft.

For that, the AI needs to remember earlier messages. `chat()` below does exactly what `ask()` does, but keeps a running list of the conversation so far.

In [ ]:
conversation = []  # keeps growing as we talk

def chat(message):
    """Send a message, remembering everything said before."""
    conversation.append({"role": "user", "content": message})
    response = client.chat.completions.create(model=MODEL, messages=conversation)
    reply = response.choices[0].message.content
    conversation.append({"role": "assistant", "content": reply})
    return reply

print(chat("Write a Python function that reads a CSV file and returns the rows as a list."))

In [ ]:
# Follow-up: ask it to improve what it just wrote
print(chat("That is good, but make the error handling more robust."))

In [ ]:
# Another follow-up
print(chat("Now add type hints to every function."))

Notice each follow-up only mentioned the *change* — no need to repeat the whole request, because `chat()` remembers.

**Exercise:** Add one more `chat(...)` cell below with your own follow-up request.

In [ ]:
print(chat("Your follow-up request here"))